In [14]:
import torch
from torch.utils.data import DataLoader,Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [15]:
torch.manual_seed(42)

In [16]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [17]:

df=pd.read_excel("fmnist_small.xlsx")
df.head(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [38]:
x=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [39]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ..., 16,  0,  0]])

In [40]:
y_train

array([7, 6, 7, ..., 0, 8, 6])

In [41]:
x_test

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  1,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ..., 51,  0,  0]])

In [42]:
y_test

array([7, 0, 5, ..., 7, 1, 2])

In [43]:
x_train=x_train/255.0
x_test=x_test/255.0

In [44]:
x_train

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.0627451, 0.       ,
        0.       ]])

In [45]:
class custom(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [46]:
train_dataset=custom(x_train,y_train)


In [47]:
train_dataset[0]

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0

In [48]:
test_dataset=custom(x_test,y_test)

In [49]:
test_dataset

In [50]:
train=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [51]:
class mynn(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model=nn.Sequential(
        nn.Linear(num_features,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,10)

    )
  def forward(self,x):
    return self.model(x)

In [52]:
epochs=100
learning_rate=0.1

In [53]:
model=mynn(x_train.shape[1])
model.to(device)
criterion=nn.CrossEntropyLoss()
optimiser=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [54]:
for epoch in range(epochs):
  total_loss=0
  for batch_features,batch_labels in train:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
    output= model(batch_features)
    loss=criterion(output,batch_labels)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    total_loss+=loss.item()

  print(f"epoch;{epoch+1},Loss:{total_loss/len(train)}")



epoch;1,Loss:0.9829786554972331
epoch;2,Loss:0.6794186288118362
epoch;3,Loss:0.6064489726225535
epoch;4,Loss:0.5860326794783274
epoch;5,Loss:0.5376663250724475
epoch;6,Loss:0.5021167414387068
epoch;7,Loss:0.47537442574898403
epoch;8,Loss:0.46131038477023445
epoch;9,Loss:0.43940632035334903
epoch;10,Loss:0.42611689736445746
epoch;11,Loss:0.40816969603300096
epoch;12,Loss:0.3916580459475517
epoch;13,Loss:0.374651267627875
epoch;14,Loss:0.36774372071027756
epoch;15,Loss:0.3646094666918119
epoch;16,Loss:0.3411906709273656
epoch;17,Loss:0.3333865141868591
epoch;18,Loss:0.3245224758982658
epoch;19,Loss:0.32092246477802594
epoch;20,Loss:0.30957318166891734
epoch;21,Loss:0.31257038553555805
epoch;22,Loss:0.2871268806606531
epoch;23,Loss:0.29130439132452013
epoch;24,Loss:0.29096027264992397
epoch;25,Loss:0.26831667552391686
epoch;26,Loss:0.2850479318201542
epoch;27,Loss:0.2671313454210758
epoch;28,Loss:0.2631259806205829
epoch;29,Loss:0.26559100886185966
epoch;30,Loss:0.2553776003917058
epoch;3

In [55]:
model.eval()

mynn(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [56]:
total=0
correct=0
with torch.no_grad():
  for batch_features,batch_labels in test:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
    output= model(batch_features)
    _, predicted=torch.max(output,1)
    total=total+batch_labels.shape[0]
    correct+=(predicted==batch_labels).sum().item()
print(correct)
print("Accuracy:", correct / total)


991
Accuracy: 0.8258333333333333
